# Cleaning Messy Chicago Traffic Crashes Dataset

In [1]:
import datetime

import numpy as np
import pandas as pd
from pathlib import Path
import importlib

import library

importlib.reload(library)

log = library.logger.getLogger(__name__)

## Get the path to the CSV dataset

In [2]:
path = Path().cwd().joinpath('chicago_crashes_dataset', 'Traffic_Crashes.csv')

Get the encoding of the CSV file (non UTF-8)

In [3]:
# Already know what the encoding is, takes a long time to detect encoding because of file size
encoding = 'ascii' or library.path.detect_encoding(path)
log.debug(f'Detecting Encoding {encoding}')

Detecting Encoding ascii


In [4]:
df = pd.read_csv(path, sep=',', encoding=encoding)

## Getting the general info to see what we're working with

In [5]:
log.info(f'General Info {df.info()}')
log.info(f'General Info {df.shape}')

General Info None
General Info (984012, 48)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 984012 entries, 0 to 984011
Data columns (total 48 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   CRASH_RECORD_ID                984012 non-null  object 
 1   CRASH_DATE_EST_I               71715 non-null   object 
 2   CRASH_DATE                     984012 non-null  object 
 3   POSTED_SPEED_LIMIT             984012 non-null  int64  
 4   TRAFFIC_CONTROL_DEVICE         984012 non-null  object 
 5   DEVICE_CONDITION               984012 non-null  object 
 6   WEATHER_CONDITION              984012 non-null  object 
 7   LIGHTING_CONDITION             984012 non-null  object 
 8   FIRST_CRASH_TYPE               984012 non-null  object 
 9   TRAFFICWAY_TYPE                984012 non-null  object 
 10  LANE_CNT                       199029 non-null  float64
 11  ALIGNMENT                      984012 non-null  object 
 12  ROADWAY_SURFACE_COND          

### Checking for null indices and rows

In [6]:
na_items = pd.isnull(df).sum()
log.debug(f'na_items {na_items}')
null_indices = library.tools.get_null_indices(df)
null_columns, null_rows = library.tools.get_all_nulls(df)
log.info(f'Null indices {len(df.columns)}/{len(null_indices)}\n{null_indices}')
log.info(f'Null columns {null_columns}')
log.info(f'Null rows {null_rows}')

na_items CRASH_RECORD_ID                       0
CRASH_DATE_EST_I                 912297
CRASH_DATE                            0
POSTED_SPEED_LIMIT                    0
TRAFFIC_CONTROL_DEVICE                0
DEVICE_CONDITION                      0
WEATHER_CONDITION                     0
LIGHTING_CONDITION                    0
FIRST_CRASH_TYPE                      0
TRAFFICWAY_TYPE                       0
LANE_CNT                         784983
ALIGNMENT                             0
ROADWAY_SURFACE_COND                  0
ROAD_DEFECT                           0
REPORT_TYPE                       32224
CRASH_TYPE                            0
INTERSECTION_RELATED_I           757867
NOT_RIGHT_OF_WAY_I               939693
HIT_AND_RUN_I                    675242
DAMAGE                                0
DATE_POLICE_NOTIFIED                  0
PRIM_CONTRIBUTORY_CAUSE               0
SEC_CONTRIBUTORY_CAUSE                0
STREET_NO                             0
STREET_DIRECTION               

## No rows or columns that are ALL null. Moving onto cleaning columns.

## Creating a backup

In [7]:
df_cp1 = df.copy()

### Cleaning 'CRASH_RECORD_ID' column
Dropping column because we don't need the ID

In [8]:
df_cp1 = df_cp1.drop('CRASH_RECORD_ID', axis=1)

In [9]:
df_cp1.columns

Index(['CRASH_DATE_EST_I', 'CRASH_DATE', 'POSTED_SPEED_LIMIT',
       'TRAFFIC_CONTROL_DEVICE', 'DEVICE_CONDITION', 'WEATHER_CONDITION',
       'LIGHTING_CONDITION', 'FIRST_CRASH_TYPE', 'TRAFFICWAY_TYPE', 'LANE_CNT',
       'ALIGNMENT', 'ROADWAY_SURFACE_COND', 'ROAD_DEFECT', 'REPORT_TYPE',
       'CRASH_TYPE', 'INTERSECTION_RELATED_I', 'NOT_RIGHT_OF_WAY_I',
       'HIT_AND_RUN_I', 'DAMAGE', 'DATE_POLICE_NOTIFIED',
       'PRIM_CONTRIBUTORY_CAUSE', 'SEC_CONTRIBUTORY_CAUSE', 'STREET_NO',
       'STREET_DIRECTION', 'STREET_NAME', 'BEAT_OF_OCCURRENCE',
       'PHOTOS_TAKEN_I', 'STATEMENTS_TAKEN_I', 'DOORING_I', 'WORK_ZONE_I',
       'WORK_ZONE_TYPE', 'WORKERS_PRESENT_I', 'NUM_UNITS',
       'MOST_SEVERE_INJURY', 'INJURIES_TOTAL', 'INJURIES_FATAL',
       'INJURIES_INCAPACITATING', 'INJURIES_NON_INCAPACITATING',
       'INJURIES_REPORTED_NOT_EVIDENT', 'INJURIES_NO_INDICATION',
       'INJURIES_UNKNOWN', 'CRASH_HOUR', 'CRASH_DAY_OF_WEEK', 'CRASH_MONTH',
       'LATITUDE', 'LONGITUDE', 'LOCAT

## Checking for Clean columns

In [10]:
# Create a null and non_null columns list
non_null_columns = []
null_columns = df_cp1.columns.tolist()

for column in df_cp1.columns:
    nulls = library.tools.isnull(df_cp1[column])

    # If the Series is empty it means we found no nulls
    # Remove the column from the null_columns and add to
    # the non_null_columns
    if nulls.empty:
        null_columns.remove(column)
        non_null_columns.append(column)

non_null_columns

['CRASH_DATE',
 'POSTED_SPEED_LIMIT',
 'TRAFFIC_CONTROL_DEVICE',
 'DEVICE_CONDITION',
 'WEATHER_CONDITION',
 'LIGHTING_CONDITION',
 'FIRST_CRASH_TYPE',
 'TRAFFICWAY_TYPE',
 'ALIGNMENT',
 'ROADWAY_SURFACE_COND',
 'ROAD_DEFECT',
 'CRASH_TYPE',
 'DAMAGE',
 'DATE_POLICE_NOTIFIED',
 'PRIM_CONTRIBUTORY_CAUSE',
 'SEC_CONTRIBUTORY_CAUSE',
 'STREET_NO',
 'NUM_UNITS',
 'CRASH_HOUR',
 'CRASH_DAY_OF_WEEK',
 'CRASH_MONTH']

In [11]:
null_columns

['CRASH_DATE_EST_I',
 'LANE_CNT',
 'REPORT_TYPE',
 'INTERSECTION_RELATED_I',
 'NOT_RIGHT_OF_WAY_I',
 'HIT_AND_RUN_I',
 'STREET_DIRECTION',
 'STREET_NAME',
 'BEAT_OF_OCCURRENCE',
 'PHOTOS_TAKEN_I',
 'STATEMENTS_TAKEN_I',
 'DOORING_I',
 'WORK_ZONE_I',
 'WORK_ZONE_TYPE',
 'WORKERS_PRESENT_I',
 'MOST_SEVERE_INJURY',
 'INJURIES_TOTAL',
 'INJURIES_FATAL',
 'INJURIES_INCAPACITATING',
 'INJURIES_NON_INCAPACITATING',
 'INJURIES_REPORTED_NOT_EVIDENT',
 'INJURIES_NO_INDICATION',
 'INJURIES_UNKNOWN',
 'LATITUDE',
 'LONGITUDE',
 'LOCATION']

## Cleaning nan columns

In [12]:
# Dictionary for reference stored with filler values to replace NaN values with
filler_values = {}

# Dictionary to store any columns that mostly have NaN values. Will need to look at these closer.
high_risk = {}

for na_col in null_columns:
    try:
        filler_value = library.tools.get_filler_value(df_cp1[na_col])
        log.debug(f'{na_col} ({df_cp1[na_col].dtypes}): {filler_value}')

        # Update filler_values dictionary
        filler_values[na_col] = filler_value

        # If the column is not a high risk column, update NaN indices with filler value
        if not library.tools.series_risk(df_cp1, df_cp1[na_col]):
            df_cp1.loc[:, na_col] = df_cp1[na_col].fillna(filler_value)
        else:
            # Need to look closer at these
            high_risk[na_col] = library.tools.series_err_rate(df_cp1, df_cp1[na_col])
    except KeyError:
        pass

CRASH_DATE_EST_I (object): Not Available
LANE_CNT (float64): 1191626.0
REPORT_TYPE (object): Not Available
INTERSECTION_RELATED_I (object): Not Available
NOT_RIGHT_OF_WAY_I (object): Not Available
HIT_AND_RUN_I (object): Not Available
STREET_DIRECTION (object): Not Available
STREET_NAME (object): Not Available
BEAT_OF_OCCURRENCE (float64): 6101.0
PHOTOS_TAKEN_I (object): Not Available
STATEMENTS_TAKEN_I (object): Not Available
DOORING_I (object): Not Available
WORK_ZONE_I (object): Not Available
WORK_ZONE_TYPE (object): Not Available
WORKERS_PRESENT_I (object): Not Available
MOST_SEVERE_INJURY (object): Not Available
INJURIES_TOTAL (float64): 22.0
INJURIES_FATAL (float64): 5.0
INJURIES_INCAPACITATING (float64): 11.0
INJURIES_NON_INCAPACITATING (float64): 22.0
INJURIES_REPORTED_NOT_EVIDENT (float64): 20.0
INJURIES_NO_INDICATION (float64): 62.0
INJURIES_UNKNOWN (float64): 1.0
LATITUDE (float64): 43.0227798611
LONGITUDE (float64): 1.0
LOCATION (object): Not Available


Taking a look at the high risk columns

In [13]:
high_risk

{'CRASH_DATE_EST_I': np.float64(0.9271197912220582),
 'LANE_CNT': np.float64(0.7977372227167961),
 'INTERSECTION_RELATED_I': np.float64(0.7701806482034772),
 'NOT_RIGHT_OF_WAY_I': np.float64(0.9549609151107913),
 'PHOTOS_TAKEN_I': np.float64(0.9859005784482303),
 'STATEMENTS_TAKEN_I': np.float64(0.9764108567781694),
 'DOORING_I': np.float64(0.9968150794909005),
 'WORK_ZONE_I': np.float64(0.9945356357442795),
 'WORK_ZONE_TYPE': np.float64(0.9958191566769511),
 'WORKERS_PRESENT_I': np.float64(0.9985965618305468)}

Removing data that is not useable

In [14]:
for col in high_risk:
    df_cp1 = df_cp1.drop(col, axis=1)

In [15]:
df_cp1.columns

Index(['CRASH_DATE', 'POSTED_SPEED_LIMIT', 'TRAFFIC_CONTROL_DEVICE',
       'DEVICE_CONDITION', 'WEATHER_CONDITION', 'LIGHTING_CONDITION',
       'FIRST_CRASH_TYPE', 'TRAFFICWAY_TYPE', 'ALIGNMENT',
       'ROADWAY_SURFACE_COND', 'ROAD_DEFECT', 'REPORT_TYPE', 'CRASH_TYPE',
       'HIT_AND_RUN_I', 'DAMAGE', 'DATE_POLICE_NOTIFIED',
       'PRIM_CONTRIBUTORY_CAUSE', 'SEC_CONTRIBUTORY_CAUSE', 'STREET_NO',
       'STREET_DIRECTION', 'STREET_NAME', 'BEAT_OF_OCCURRENCE', 'NUM_UNITS',
       'MOST_SEVERE_INJURY', 'INJURIES_TOTAL', 'INJURIES_FATAL',
       'INJURIES_INCAPACITATING', 'INJURIES_NON_INCAPACITATING',
       'INJURIES_REPORTED_NOT_EVIDENT', 'INJURIES_NO_INDICATION',
       'INJURIES_UNKNOWN', 'CRASH_HOUR', 'CRASH_DAY_OF_WEEK', 'CRASH_MONTH',
       'LATITUDE', 'LONGITUDE', 'LOCATION'],
      dtype='object')

## Creating a backup

In [16]:
df_cp2 = df_cp1.copy()

In [17]:
df_cp2.columns

Index(['CRASH_DATE', 'POSTED_SPEED_LIMIT', 'TRAFFIC_CONTROL_DEVICE',
       'DEVICE_CONDITION', 'WEATHER_CONDITION', 'LIGHTING_CONDITION',
       'FIRST_CRASH_TYPE', 'TRAFFICWAY_TYPE', 'ALIGNMENT',
       'ROADWAY_SURFACE_COND', 'ROAD_DEFECT', 'REPORT_TYPE', 'CRASH_TYPE',
       'HIT_AND_RUN_I', 'DAMAGE', 'DATE_POLICE_NOTIFIED',
       'PRIM_CONTRIBUTORY_CAUSE', 'SEC_CONTRIBUTORY_CAUSE', 'STREET_NO',
       'STREET_DIRECTION', 'STREET_NAME', 'BEAT_OF_OCCURRENCE', 'NUM_UNITS',
       'MOST_SEVERE_INJURY', 'INJURIES_TOTAL', 'INJURIES_FATAL',
       'INJURIES_INCAPACITATING', 'INJURIES_NON_INCAPACITATING',
       'INJURIES_REPORTED_NOT_EVIDENT', 'INJURIES_NO_INDICATION',
       'INJURIES_UNKNOWN', 'CRASH_HOUR', 'CRASH_DAY_OF_WEEK', 'CRASH_MONTH',
       'LATITUDE', 'LONGITUDE', 'LOCATION'],
      dtype='object')

## Validating data

In [18]:
df_cp2.loc[:, 'CRASH_DATE']

0         01/14/2025 12:25:00 PM
1         05/23/2025 09:30:00 AM
2         04/05/2025 08:00:00 PM
3         05/23/2025 09:15:00 AM
4         01/14/2025 08:00:00 AM
                   ...          
984007    08/07/2025 06:40:00 PM
984008    08/16/2025 11:50:00 PM
984009    08/17/2025 02:00:00 PM
984010    08/18/2025 12:08:00 AM
984011    08/18/2025 12:24:00 AM
Name: CRASH_DATE, Length: 984012, dtype: object

Unifying proper formatting for pd.Timestamp data type

In [19]:
df_cp2.loc[:, 'POSTED_SPEED_LIMIT']

0         30
1         30
2         30
3         30
4         30
          ..
984007    30
984008    30
984009    30
984010    30
984011    30
Name: POSTED_SPEED_LIMIT, Length: 984012, dtype: int64

Quick check of values

In [20]:
minimum = df_cp2.loc[:, 'POSTED_SPEED_LIMIT'].min()
maximum = df_cp2.loc[:, 'POSTED_SPEED_LIMIT'].max()
average = df_cp2.loc[:, 'POSTED_SPEED_LIMIT'].mean()
log.info(f'Min/Max/Avg POSTED_SPEED_LIMIT {minimum}/{maximum}/{average}')

Min/Max/Avg POSTED_SPEED_LIMIT 0/99/28.423204188566807


Valid values, no need to clean this column

Checking for categories

In [21]:
df_cp2.loc[:, 'DEVICE_CONDITION'].unique()

array(['FUNCTIONING PROPERLY', 'UNKNOWN', 'NO CONTROLS',
       'FUNCTIONING IMPROPERLY', 'OTHER', 'NOT FUNCTIONING',
       'WORN REFLECTIVE MATERIAL', 'MISSING'], dtype=object)

In [22]:
df_cp2.loc[:, 'WEATHER_CONDITION'].unique()

array(['SNOW', 'UNKNOWN', 'CLEAR', 'RAIN', 'CLOUDY/OVERCAST',
       'FREEZING RAIN/DRIZZLE', 'SLEET/HAIL', 'OTHER', 'BLOWING SNOW',
       'SEVERE CROSS WIND GATE', 'FOG/SMOKE/HAZE',
       'BLOWING SAND, SOIL, DIRT'], dtype=object)

To keep with current formatting, switching 'BLOWING SAND, SOIL, DIRT' to '/'

In [23]:
df_cp2.loc[:, 'WEATHER_CONDITION'] = df_cp2.loc[:, 'WEATHER_CONDITION'].str.replace(', ', '/')
df_cp2.loc[:, 'WEATHER_CONDITION'].unique()

array(['SNOW', 'UNKNOWN', 'CLEAR', 'RAIN', 'CLOUDY/OVERCAST',
       'FREEZING RAIN/DRIZZLE', 'SLEET/HAIL', 'OTHER', 'BLOWING SNOW',
       'SEVERE CROSS WIND GATE', 'FOG/SMOKE/HAZE',
       'BLOWING SAND/SOIL/DIRT'], dtype=object)

In [24]:
df_cp2.loc[:, 'LIGHTING_CONDITION'].unique()

array(['DAYLIGHT', 'UNKNOWN', 'DARKNESS, LIGHTED ROAD', 'DARKNESS',
       'DUSK', 'DAWN'], dtype=object)

In [25]:
df_cp2.loc[:, 'FIRST_CRASH_TYPE'].unique()

array(['SIDESWIPE SAME DIRECTION', 'TURNING', 'PARKED MOTOR VEHICLE',
       'PEDALCYCLIST', 'REAR END', 'ANGLE',
       'SIDESWIPE OPPOSITE DIRECTION', 'FIXED OBJECT', 'PEDESTRIAN',
       'REAR TO SIDE', 'HEAD ON', 'REAR TO FRONT', 'OTHER OBJECT',
       'OTHER NONCOLLISION', 'REAR TO REAR', 'OVERTURNED', 'ANIMAL',
       'TRAIN'], dtype=object)

In [26]:
df_cp2['TRAFFICWAY_TYPE'].unique()

array(['DIVIDED - W/MEDIAN (NOT RAISED)', 'NOT DIVIDED', 'FOUR WAY',
       'DIVIDED - W/MEDIAN BARRIER', 'ONE-WAY', 'PARKING LOT', 'UNKNOWN',
       'T-INTERSECTION', 'OTHER', 'TRAFFIC ROUTE', 'DRIVEWAY',
       'UNKNOWN INTERSECTION TYPE', 'ALLEY', 'Y-INTERSECTION',
       'FIVE POINT, OR MORE', 'L-INTERSECTION', 'RAMP',
       'CENTER TURN LANE', 'NOT REPORTED', 'ROUNDABOUT'], dtype=object)

In [27]:
df_cp2['ALIGNMENT'].unique()

array(['STRAIGHT AND LEVEL', 'STRAIGHT ON GRADE', 'STRAIGHT ON HILLCREST',
       'CURVE, LEVEL', 'CURVE ON GRADE', 'CURVE ON HILLCREST'],
      dtype=object)

In [28]:
df_cp2['ROADWAY_SURFACE_COND'].unique()

array(['SNOW OR SLUSH', 'UNKNOWN', 'DRY', 'WET', 'ICE', 'OTHER',
       'SAND, MUD, DIRT'], dtype=object)

In [29]:
df_cp2['ROAD_DEFECT'].unique()

array(['NO DEFECTS', 'UNKNOWN', 'SHOULDER DEFECT', 'WORN SURFACE',
       'RUT, HOLES', 'OTHER', 'DEBRIS ON ROADWAY'], dtype=object)

In [30]:
df_cp2.columns

Index(['CRASH_DATE', 'POSTED_SPEED_LIMIT', 'TRAFFIC_CONTROL_DEVICE',
       'DEVICE_CONDITION', 'WEATHER_CONDITION', 'LIGHTING_CONDITION',
       'FIRST_CRASH_TYPE', 'TRAFFICWAY_TYPE', 'ALIGNMENT',
       'ROADWAY_SURFACE_COND', 'ROAD_DEFECT', 'REPORT_TYPE', 'CRASH_TYPE',
       'HIT_AND_RUN_I', 'DAMAGE', 'DATE_POLICE_NOTIFIED',
       'PRIM_CONTRIBUTORY_CAUSE', 'SEC_CONTRIBUTORY_CAUSE', 'STREET_NO',
       'STREET_DIRECTION', 'STREET_NAME', 'BEAT_OF_OCCURRENCE', 'NUM_UNITS',
       'MOST_SEVERE_INJURY', 'INJURIES_TOTAL', 'INJURIES_FATAL',
       'INJURIES_INCAPACITATING', 'INJURIES_NON_INCAPACITATING',
       'INJURIES_REPORTED_NOT_EVIDENT', 'INJURIES_NO_INDICATION',
       'INJURIES_UNKNOWN', 'CRASH_HOUR', 'CRASH_DAY_OF_WEEK', 'CRASH_MONTH',
       'LATITUDE', 'LONGITUDE', 'LOCATION'],
      dtype='object')

In [31]:
df_cp2['REPORT_TYPE'].unique()

array(['ON SCENE', 'NOT ON SCENE (DESK REPORT)', 'Not Available',
       'AMENDED'], dtype=object)

In [32]:
df_cp2['CRASH_TYPE'].unique()

array(['NO INJURY / DRIVE AWAY', 'INJURY AND / OR TOW DUE TO CRASH'],
      dtype=object)

In [33]:
df_cp2['HIT_AND_RUN_I'].unique()

array(['Y', 'Not Available', 'N'], dtype=object)

In [34]:
df_cp2['DAMAGE'].unique()

array(['$501 - $1,500', 'OVER $1,500', '$500 OR LESS'], dtype=object)

In [35]:
df_cp2['DATE_POLICE_NOTIFIED'].unique()

array(['01/14/2025 12:38:00 PM', '05/27/2025 10:40:00 AM',
       '04/05/2025 09:23:00 PM', ..., '08/16/2025 11:55:00 PM',
       '08/18/2025 12:09:00 AM', '08/18/2025 12:28:00 AM'],
      shape=(744929,), dtype=object)

In [36]:
df_cp2['PRIM_CONTRIBUTORY_CAUSE'].unique()

array(['IMPROPER TURNING/NO SIGNAL', 'IMPROPER OVERTAKING/PASSING',
       'UNABLE TO DETERMINE', 'FOLLOWING TOO CLOSELY',
       'DISREGARDING TRAFFIC SIGNALS', 'IMPROPER LANE USAGE',
       'FAILING TO YIELD RIGHT-OF-WAY', 'NOT APPLICABLE',
       'UNDER THE INFLUENCE OF ALCOHOL/DRUGS (USE WHEN ARREST IS EFFECTED)',
       'FAILING TO REDUCE SPEED TO AVOID CRASH',
       'DRIVING SKILLS/KNOWLEDGE/EXPERIENCE', 'WEATHER',
       'DISREGARDING STOP SIGN', 'IMPROPER BACKING',
       'OPERATING VEHICLE IN ERRATIC, RECKLESS, CARELESS, NEGLIGENT OR AGGRESSIVE MANNER',
       'TURNING RIGHT ON RED', 'PHYSICAL CONDITION OF DRIVER',
       'EQUIPMENT - VEHICLE CONDITION', 'DRIVING ON WRONG SIDE/WRONG WAY',
       'DISREGARDING ROAD MARKINGS',
       'EVASIVE ACTION DUE TO ANIMAL, OBJECT, NONMOTORIST',
       'ROAD ENGINEERING/SURFACE/MARKING DEFECTS',
       'VISION OBSCURED (SIGNS, TREE LIMBS, BUILDINGS, ETC.)',
       'DISTRACTION - FROM OUTSIDE VEHICLE',
       'DISTRACTION - OTHER ELECTRON

In [37]:
df_cp2['SEC_CONTRIBUTORY_CAUSE'].unique()

array(['IMPROPER OVERTAKING/PASSING', 'UNABLE TO DETERMINE',
       'FOLLOWING TOO CLOSELY', 'NOT APPLICABLE',
       'FAILING TO YIELD RIGHT-OF-WAY',
       'DRIVING SKILLS/KNOWLEDGE/EXPERIENCE', 'WEATHER',
       'FAILING TO REDUCE SPEED TO AVOID CRASH',
       'PHYSICAL CONDITION OF DRIVER', 'IMPROPER LANE USAGE',
       'OPERATING VEHICLE IN ERRATIC, RECKLESS, CARELESS, NEGLIGENT OR AGGRESSIVE MANNER',
       'IMPROPER BACKING', 'DISREGARDING STOP SIGN',
       'UNDER THE INFLUENCE OF ALCOHOL/DRUGS (USE WHEN ARREST IS EFFECTED)',
       'IMPROPER TURNING/NO SIGNAL', 'ROAD CONSTRUCTION/MAINTENANCE',
       'DRIVING ON WRONG SIDE/WRONG WAY', 'DISREGARDING TRAFFIC SIGNALS',
       'EVASIVE ACTION DUE TO ANIMAL, OBJECT, NONMOTORIST',
       'EQUIPMENT - VEHICLE CONDITION',
       'CELL PHONE USE OTHER THAN TEXTING',
       'VISION OBSCURED (SIGNS, TREE LIMBS, BUILDINGS, ETC.)',
       'DISTRACTION - FROM OUTSIDE VEHICLE',
       'ROAD ENGINEERING/SURFACE/MARKING DEFECTS', 'TEXTING',
  

In [38]:
df_cp2['STREET_NO']

0         6352
1         3555
2         1005
3         2901
4         4001
          ... 
984007    7137
984008    5001
984009     555
984010    2034
984011    9620
Name: STREET_NO, Length: 984012, dtype: int64

In [39]:
df_cp2['STREET_DIRECTION'].unique()

array(['N', 'W', 'S', 'E', 'Not Available'], dtype=object)

In [40]:
df_cp2['STREET_DIRECTION'].value_counts()['Not Available']

np.int64(4)

In [41]:
df_cp2['BEAT_OF_OCCURRENCE']

0         2433.0
1         1921.0
2         1121.0
3         1211.0
4         2211.0
           ...  
984007    2411.0
984008     813.0
984009     915.0
984010    1424.0
984011     432.0
Name: BEAT_OF_OCCURRENCE, Length: 984012, dtype: float64

In [42]:
df_cp2['NUM_UNITS'].unique()

array([ 2,  1,  5,  3,  4,  6,  7, 10,  8, 14, 12, 18, 11,  9, 16, 13, 15])

In [43]:
df_cp2['MOST_SEVERE_INJURY'].unique()

array(['NO INDICATION OF INJURY', 'NONINCAPACITATING INJURY',
       'REPORTED, NOT EVIDENT', 'INCAPACITATING INJURY', 'FATAL',
       'Not Available'], dtype=object)

In [44]:
df_cp2['INJURIES_TOTAL'].unique()

array([ 0.,  1.,  2.,  3.,  5.,  6.,  4., 22.,  7., 11.,  8.,  9., 19.,
       15., 10., 21., 17., 12., 14., 13., 16.])

In [45]:
df_cp2['INJURIES_FATAL'].unique()

array([0., 1., 5., 2., 3., 4.])

In [46]:
df_cp2['INJURIES_INCAPACITATING'].unique()

array([ 0.,  1.,  2., 11.,  3.,  5.,  4.,  6.,  7., 10.,  8.])

In [47]:
sorted(df_cp2['INJURIES_NON_INCAPACITATING'].astype(int).unique())

[np.int64(0),
 np.int64(1),
 np.int64(2),
 np.int64(3),
 np.int64(4),
 np.int64(5),
 np.int64(6),
 np.int64(7),
 np.int64(8),
 np.int64(9),
 np.int64(10),
 np.int64(11),
 np.int64(12),
 np.int64(13),
 np.int64(14),
 np.int64(15),
 np.int64(16),
 np.int64(18),
 np.int64(19),
 np.int64(21),
 np.int64(22)]

In [48]:
sorted(df_cp2['INJURIES_REPORTED_NOT_EVIDENT'].astype(int).unique())

[np.int64(0),
 np.int64(1),
 np.int64(2),
 np.int64(3),
 np.int64(4),
 np.int64(5),
 np.int64(6),
 np.int64(7),
 np.int64(8),
 np.int64(9),
 np.int64(10),
 np.int64(11),
 np.int64(15),
 np.int64(19),
 np.int64(20)]

In [49]:
sorted(df_cp2['INJURIES_NO_INDICATION'].astype(int).unique())

[np.int64(0),
 np.int64(1),
 np.int64(2),
 np.int64(3),
 np.int64(4),
 np.int64(5),
 np.int64(6),
 np.int64(7),
 np.int64(8),
 np.int64(9),
 np.int64(10),
 np.int64(11),
 np.int64(12),
 np.int64(13),
 np.int64(14),
 np.int64(15),
 np.int64(16),
 np.int64(17),
 np.int64(18),
 np.int64(19),
 np.int64(20),
 np.int64(21),
 np.int64(22),
 np.int64(23),
 np.int64(24),
 np.int64(25),
 np.int64(26),
 np.int64(27),
 np.int64(28),
 np.int64(29),
 np.int64(30),
 np.int64(31),
 np.int64(32),
 np.int64(33),
 np.int64(34),
 np.int64(35),
 np.int64(36),
 np.int64(37),
 np.int64(38),
 np.int64(39),
 np.int64(40),
 np.int64(41),
 np.int64(42),
 np.int64(43),
 np.int64(45),
 np.int64(46),
 np.int64(48),
 np.int64(49),
 np.int64(50),
 np.int64(61),
 np.int64(62)]

In [50]:
sorted(df_cp2['INJURIES_UNKNOWN'].astype(int).unique())

[np.int64(0), np.int64(1)]

In [51]:
df_cp2['CRASH_HOUR'].unique() > 24

array([False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False])

In [52]:
df_cp2['CRASH_DAY_OF_WEEK'].unique() > 7

array([False, False, False, False, False, False, False])

In [53]:
df_cp2['CRASH_MONTH'].unique() > 12

array([False, False, False, False, False, False, False, False, False,
       False, False, False])

In [54]:
df_cp2['LATITUDE']

0         41.997808
1         41.946529
2         41.899325
3         41.902793
4         41.691207
            ...    
984007    42.012120
984008    41.778225
984009    41.838002
984010    41.904205
984011    41.720684
Name: LATITUDE, Length: 984012, dtype: float64

In [55]:
df_cp2['LONGITUDE']

0        -87.655770
1        -87.688106
2        -87.715074
3        -87.699412
4        -87.720555
            ...    
984007   -87.690100
984008   -87.747206
984009   -87.641182
984010   -87.678698
984011   -87.536644
Name: LONGITUDE, Length: 984012, dtype: float64

In [56]:
df_cp2['LOCATION']

0         POINT (-87.655770494712 41.997807727633)
1         POINT (-87.688106391039 41.946529480518)
2         POINT (-87.715074373867 41.899324573751)
3         POINT (-87.699412181285 41.902792968177)
4         POINT (-87.720554863466 41.691206664451)
                            ...                   
984007    POINT (-87.690099690923 42.012120373394)
984008    POINT (-87.747206294662 41.778224612839)
984009    POINT (-87.641182383332 41.838001538617)
984010    POINT (-87.678698087004 41.904205053083)
984011    POINT (-87.536644107447 41.720683731954)
Name: LOCATION, Length: 984012, dtype: object

## Creating lists todo

In [57]:
to_consolidate = [
    'WEATHER_CONDITION',
    'TRAFFIC_CONTROL_DEVICE',
    'DEVICE_CONDITION',
    'FIRST_CRASH_TYPE', #?
    'ALIGNMENT,'
    'TRAFFICWAY_TYPE',
    'ROADWAY_SURFACE_COND',
    'ROAD_DEFECT',
    'PRIM_CONTRIBUTORY_CAUSE',
    'SEC_CONTRIBUTORY_CAUSE',
    'ROADWAY_SURFACE_COND'

]

to_bools_manual = [
    'REPORT_TYPE', # Change to ON_SCENE
    'CRASH_TYPE', # Change to REPORTABLE
    'HIT_AND_RUN_I'
]

to_ints_manual = [
    'DAMAGE'
]
to_ints = [
    'INJURIES_TOTAL',
    'INJURIES_FATAL',
    'INJURIES_INCAPACITATING',
    'INJURIES_NON_INCAPACITATING',
    'INJURIES_REPORTED_NOT_EVIDENT',
    'INJURIES_NO_INDICATION'
]

to_datetime = [
    'CRASH_DATE',
    'DATE_POLICE_NOTIFIED',
]

to_drop = [
    'STREET_NAME',
    'STREET_NO',
    'STREET_DIRECTION',
    'BEAT_OF_OCCURRENCE',
    'MOST_SEVERE_INJURY', # Repeated in other INJURY_* columns
    'CRASH_HOUR',  # repeated in CRASH_DATE
    'CRASH_DAY_OF_WEEK',  # repeated in CRASH_DATE
    'CRASH_MONTH',  # repeated in CRASH_DATE
    'NUM_UNITS', # Would need clarification what this means
    'LOCATION'
]

valid = [
    'POSTED_SPEED_LIMIT',
    'LIGHTING_CONDITION',
    'LATITUDE',
    'LONGITUDE',
]

keys2 = to_consolidate + to_bools_manual + to_ints_manual + to_ints + to_datetime + to_drop + valid
diff = set(df_cp2.columns).difference(set(keys2))
log.debug(f'Diff {diff}: {df_cp2.shape[1]}/{len(keys2)}')

Diff {'TRAFFICWAY_TYPE', 'INJURIES_UNKNOWN', 'ALIGNMENT'}: 37/36


## Creating a backup

In [58]:
df_cp3 = df_cp2.copy()

## Dropping unneeded Columns

In [59]:
df_cp3 = df_cp3.drop(to_drop, axis=1)

## Converting Datetime columns

In [60]:
for column in to_datetime:
    df_cp3[column] = pd.to_datetime(df_cp3.loc[:, column], format='%m/%d/%Y %I:%M:%S %p')

In [61]:
df_cp3['DATE_POLICE_NOTIFIED']

0        2025-01-14 12:38:00
1        2025-05-27 10:40:00
2        2025-04-05 21:23:00
3        2025-05-23 09:16:00
4        2025-01-14 08:45:00
                 ...        
984007   2025-08-08 15:30:00
984008   2025-08-16 23:55:00
984009   2025-08-18 15:30:00
984010   2025-08-18 00:09:00
984011   2025-08-18 00:28:00
Name: DATE_POLICE_NOTIFIED, Length: 984012, dtype: datetime64[ns]

## Converting Ints columns

In [62]:
df_cp3['DAMAGE']

0         $501 - $1,500
1           OVER $1,500
2           OVER $1,500
3          $500 OR LESS
4           OVER $1,500
              ...      
984007    $501 - $1,500
984008      OVER $1,500
984009     $500 OR LESS
984010      OVER $1,500
984011      OVER $1,500
Name: DAMAGE, Length: 984012, dtype: object

In [63]:
# Converting to ints
df_cp3['DAMAGE'] = df_cp3['DAMAGE'].replace('OVER $1,500', 1500)
df_cp3['DAMAGE'] = df_cp3['DAMAGE'].replace('$501 - $1,500', 1499)
df_cp3['DAMAGE'] = df_cp3['DAMAGE'].replace('$500 OR LESS', 499)
df_cp3['DAMAGE'].astype(int)

/var/folders/_9/nyjys43x4nv6dxyk3y75vzh80000gn/T/ipykernel_5468/582650866.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_cp3['DAMAGE'] = df_cp3['DAMAGE'].replace('$500 OR LESS', 499)


0         1499
1         1500
2         1500
3          499
4         1500
          ... 
984007    1499
984008    1500
984009     499
984010    1500
984011    1500
Name: DAMAGE, Length: 984012, dtype: int64

In [64]:
for column in to_ints:
    df_cp3[column] = df_cp3[column].astype(int)

## Converting Bools columns
-1 for Not Available
0 for False
1 for True

Renaming 'REPORT_TYPE' to 'ON_SCENE'

In [65]:
df_cp3['REPORT_TYPE'].unique()

array(['ON SCENE', 'NOT ON SCENE (DESK REPORT)', 'Not Available',
       'AMENDED'], dtype=object)

Change to ON_SCENE

In [66]:
df_cp3.loc[:, 'ON_SCENE'] = df_cp3['REPORT_TYPE'].replace(
    {
        'ON SCENE': 1,
        'NOT ON SCENE (DESK REPORT)': 0,
        'Not Available': -1,
     }
)

In [67]:
df_cp3 = df_cp3.drop('REPORT_TYPE', axis=1)
df_cp3.loc[:, 'ON_SCENE']

0          1
1          0
2          1
3          1
4          0
          ..
984007     0
984008     1
984009     0
984010    -1
984011     1
Name: ON_SCENE, Length: 984012, dtype: object

Renaming to REPORTABLE

In [68]:
df_cp3['CRASH_TYPE'].unique()

array(['NO INJURY / DRIVE AWAY', 'INJURY AND / OR TOW DUE TO CRASH'],
      dtype=object)

In [69]:
df_cp3.loc[:, 'REPORTABLE'] = df_cp3['CRASH_TYPE'].replace(
    {
        'NO INJURY / DRIVE AWAY': False,
        'INJURY AND / OR TOW DUE TO CRASH': True,
     }
)

/var/folders/_9/nyjys43x4nv6dxyk3y75vzh80000gn/T/ipykernel_5468/852357370.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_cp3.loc[:, 'REPORTABLE'] = df_cp3['CRASH_TYPE'].replace(


In [70]:
df_cp3 = df_cp3.drop('CRASH_TYPE', axis=1)
df_cp3.loc[:, 'REPORTABLE']

0         False
1         False
2         False
3          True
4         False
          ...  
984007    False
984008     True
984009    False
984010     True
984011     True
Name: REPORTABLE, Length: 984012, dtype: bool

In [71]:
df_cp3['HIT_AND_RUN_I'].unique()

array(['Y', 'Not Available', 'N'], dtype=object)

In [72]:
df_cp3.loc[:, 'HIT_AND_RUN'] = df_cp3['HIT_AND_RUN_I'].replace(
    {
        'Not Available': -1,
        'N': 0,
        'Y': 1,
     }
)
df_cp3 = df_cp3.drop('HIT_AND_RUN_I', axis=1)

/var/folders/_9/nyjys43x4nv6dxyk3y75vzh80000gn/T/ipykernel_5468/261705232.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_cp3.loc[:, 'HIT_AND_RUN'] = df_cp3['HIT_AND_RUN_I'].replace(


## Creating a backup

In [73]:
df_cp4 = df_cp3.copy()

## Converting Consolidate column
A manual process

In [74]:
df_cp4['WEATHER_CONDITION'].unique()

array(['SNOW', 'UNKNOWN', 'CLEAR', 'RAIN', 'CLOUDY/OVERCAST',
       'FREEZING RAIN/DRIZZLE', 'SLEET/HAIL', 'OTHER', 'BLOWING SNOW',
       'SEVERE CROSS WIND GATE', 'FOG/SMOKE/HAZE',
       'BLOWING SAND/SOIL/DIRT'], dtype=object)

In [75]:
df_cp4['TRAFFIC_CONTROL_DEVICE'].unique()

array(['TRAFFIC SIGNAL', 'STOP SIGN/FLASHER', 'NO CONTROLS', 'UNKNOWN',
       'OTHER', 'LANE USE MARKING', 'PEDESTRIAN CROSSING SIGN',
       'OTHER REG. SIGN', 'OTHER WARNING SIGN', 'YIELD', 'SCHOOL ZONE',
       'FLASHING CONTROL SIGNAL', 'DELINEATORS', 'NO PASSING',
       'RAILROAD CROSSING GATE', 'POLICE/FLAGMAN', 'RR CROSSING SIGN',
       'BICYCLE CROSSING SIGN', 'OTHER RAILROAD CROSSING'], dtype=object)

In [76]:
df_cp4['DEVICE_CONDITION'].unique()

array(['FUNCTIONING PROPERLY', 'UNKNOWN', 'NO CONTROLS',
       'FUNCTIONING IMPROPERLY', 'OTHER', 'NOT FUNCTIONING',
       'WORN REFLECTIVE MATERIAL', 'MISSING'], dtype=object)

In [77]:
df_cp4['DEVICE_FUNCTIONING'] = df_cp4['DEVICE_CONDITION'].replace(
    {
        'FUNCTIONING PROPERLY': True,
        'UNKNOWN': False,
        'NO CONTROLS': False,
        'FUNCTIONING IMPROPERLY': False,
        'OTHER': False,
        'NOT FUNCTIONING': False,
        'WORN REFLECTIVE MATERIAL': False,
        'MISSING': False,
    }
)
df_cp4 = df_cp4.drop('DEVICE_CONDITION', axis=1)

/var/folders/_9/nyjys43x4nv6dxyk3y75vzh80000gn/T/ipykernel_5468/1951298382.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_cp4['DEVICE_FUNCTIONING'] = df_cp4['DEVICE_CONDITION'].replace(


In [78]:
df_cp4['FIRST_CRASH_TYPE'].unique()

array(['SIDESWIPE SAME DIRECTION', 'TURNING', 'PARKED MOTOR VEHICLE',
       'PEDALCYCLIST', 'REAR END', 'ANGLE',
       'SIDESWIPE OPPOSITE DIRECTION', 'FIXED OBJECT', 'PEDESTRIAN',
       'REAR TO SIDE', 'HEAD ON', 'REAR TO FRONT', 'OTHER OBJECT',
       'OTHER NONCOLLISION', 'REAR TO REAR', 'OVERTURNED', 'ANIMAL',
       'TRAIN'], dtype=object)

In [79]:
# collision, fixed, movement, nonmotorist
df_cp4['CRASH_TYPE'] = df_cp4['FIRST_CRASH_TYPE'].replace(
    {
        'SIDESWIPE SAME DIRECTION': 'Collision',
        'SIDESWIPE OPPOSITE DIRECTION': 'Collision',
        'REAR END': 'Collision',
        'REAR TO SIDE': 'Collision',
        'HEAD ON': 'Collision',
        'REAR TO FRONT': 'Collision',
        'OTHER OBJECT': 'Collision',
        'REAR TO REAR': 'Collision',

        'PARKED MOTOR VEHICLE': 'Fixed',
        'FIXED OBJECT': 'Fixed',
        'TRAIN': 'Fixed',

        'TURNING': 'Movement',
        'ANGLE': 'Movement',
        'OVERTURNED': 'Movement',

        'PEDALCYCLIST': 'Non-motorist',
        'PEDESTRIAN': 'Non-motorist',
        'ANIMAL': 'Non-motorist',

        'OTHER NONCOLLISION': 'OTHER NONCOLLISION',
    }
)
df_cp4 = df_cp4.drop('FIRST_CRASH_TYPE', axis=1)

In [80]:
df_cp4['TRAFFICWAY_TYPE'].unique()

array(['DIVIDED - W/MEDIAN (NOT RAISED)', 'NOT DIVIDED', 'FOUR WAY',
       'DIVIDED - W/MEDIAN BARRIER', 'ONE-WAY', 'PARKING LOT', 'UNKNOWN',
       'T-INTERSECTION', 'OTHER', 'TRAFFIC ROUTE', 'DRIVEWAY',
       'UNKNOWN INTERSECTION TYPE', 'ALLEY', 'Y-INTERSECTION',
       'FIVE POINT, OR MORE', 'L-INTERSECTION', 'RAMP',
       'CENTER TURN LANE', 'NOT REPORTED', 'ROUNDABOUT'], dtype=object)

In [81]:
df_cp4['ROADWAY_SURFACE_COND'].unique()

array(['SNOW OR SLUSH', 'UNKNOWN', 'DRY', 'WET', 'ICE', 'OTHER',
       'SAND, MUD, DIRT'], dtype=object)

In [82]:
df_cp4['ROAD_DEFECT'].unique()

array(['NO DEFECTS', 'UNKNOWN', 'SHOULDER DEFECT', 'WORN SURFACE',
       'RUT, HOLES', 'OTHER', 'DEBRIS ON ROADWAY'], dtype=object)

In [83]:
df_cp4['ROAD_DEFECT'] = df_cp3['ROAD_DEFECT']

In [84]:
df_cp4['ROAD_DEFECT'] = df_cp4['ROAD_DEFECT'].replace(
    {
        'NO DEFECTS': False,
        'UNKNOWN': True,
        'SHOULDER DEFECT': True,
        'WORN SURFACE': True,
        'RUT, HOLES': True,
        'OTHER': True,
        'DEBRIS ON ROADWAY': True
    }
)

/var/folders/_9/nyjys43x4nv6dxyk3y75vzh80000gn/T/ipykernel_5468/2626659515.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_cp4['ROAD_DEFECT'] = df_cp4['ROAD_DEFECT'].replace(


In [85]:
# Todo: combine these two?
df_cp4['PRIM_CONTRIBUTORY_CAUSE'].unique()

array(['IMPROPER TURNING/NO SIGNAL', 'IMPROPER OVERTAKING/PASSING',
       'UNABLE TO DETERMINE', 'FOLLOWING TOO CLOSELY',
       'DISREGARDING TRAFFIC SIGNALS', 'IMPROPER LANE USAGE',
       'FAILING TO YIELD RIGHT-OF-WAY', 'NOT APPLICABLE',
       'UNDER THE INFLUENCE OF ALCOHOL/DRUGS (USE WHEN ARREST IS EFFECTED)',
       'FAILING TO REDUCE SPEED TO AVOID CRASH',
       'DRIVING SKILLS/KNOWLEDGE/EXPERIENCE', 'WEATHER',
       'DISREGARDING STOP SIGN', 'IMPROPER BACKING',
       'OPERATING VEHICLE IN ERRATIC, RECKLESS, CARELESS, NEGLIGENT OR AGGRESSIVE MANNER',
       'TURNING RIGHT ON RED', 'PHYSICAL CONDITION OF DRIVER',
       'EQUIPMENT - VEHICLE CONDITION', 'DRIVING ON WRONG SIDE/WRONG WAY',
       'DISREGARDING ROAD MARKINGS',
       'EVASIVE ACTION DUE TO ANIMAL, OBJECT, NONMOTORIST',
       'ROAD ENGINEERING/SURFACE/MARKING DEFECTS',
       'VISION OBSCURED (SIGNS, TREE LIMBS, BUILDINGS, ETC.)',
       'DISTRACTION - FROM OUTSIDE VEHICLE',
       'DISTRACTION - OTHER ELECTRON

In [86]:
df_cp4['SEC_CONTRIBUTORY_CAUSE'].unique()

array(['IMPROPER OVERTAKING/PASSING', 'UNABLE TO DETERMINE',
       'FOLLOWING TOO CLOSELY', 'NOT APPLICABLE',
       'FAILING TO YIELD RIGHT-OF-WAY',
       'DRIVING SKILLS/KNOWLEDGE/EXPERIENCE', 'WEATHER',
       'FAILING TO REDUCE SPEED TO AVOID CRASH',
       'PHYSICAL CONDITION OF DRIVER', 'IMPROPER LANE USAGE',
       'OPERATING VEHICLE IN ERRATIC, RECKLESS, CARELESS, NEGLIGENT OR AGGRESSIVE MANNER',
       'IMPROPER BACKING', 'DISREGARDING STOP SIGN',
       'UNDER THE INFLUENCE OF ALCOHOL/DRUGS (USE WHEN ARREST IS EFFECTED)',
       'IMPROPER TURNING/NO SIGNAL', 'ROAD CONSTRUCTION/MAINTENANCE',
       'DRIVING ON WRONG SIDE/WRONG WAY', 'DISREGARDING TRAFFIC SIGNALS',
       'EVASIVE ACTION DUE TO ANIMAL, OBJECT, NONMOTORIST',
       'EQUIPMENT - VEHICLE CONDITION',
       'CELL PHONE USE OTHER THAN TEXTING',
       'VISION OBSCURED (SIGNS, TREE LIMBS, BUILDINGS, ETC.)',
       'DISTRACTION - FROM OUTSIDE VEHICLE',
       'ROAD ENGINEERING/SURFACE/MARKING DEFECTS', 'TEXTING',
  

In [87]:
x = pd.Series(df_cp4['PRIM_CONTRIBUTORY_CAUSE'].unique() + df_cp4['SEC_CONTRIBUTORY_CAUSE'].unique())
x.unique()

array(['IMPROPER TURNING/NO SIGNALIMPROPER OVERTAKING/PASSING',
       'IMPROPER OVERTAKING/PASSINGUNABLE TO DETERMINE',
       'UNABLE TO DETERMINEFOLLOWING TOO CLOSELY',
       'FOLLOWING TOO CLOSELYNOT APPLICABLE',
       'DISREGARDING TRAFFIC SIGNALSFAILING TO YIELD RIGHT-OF-WAY',
       'IMPROPER LANE USAGEDRIVING SKILLS/KNOWLEDGE/EXPERIENCE',
       'FAILING TO YIELD RIGHT-OF-WAYWEATHER',
       'NOT APPLICABLEFAILING TO REDUCE SPEED TO AVOID CRASH',
       'UNDER THE INFLUENCE OF ALCOHOL/DRUGS (USE WHEN ARREST IS EFFECTED)PHYSICAL CONDITION OF DRIVER',
       'FAILING TO REDUCE SPEED TO AVOID CRASHIMPROPER LANE USAGE',
       'DRIVING SKILLS/KNOWLEDGE/EXPERIENCEOPERATING VEHICLE IN ERRATIC, RECKLESS, CARELESS, NEGLIGENT OR AGGRESSIVE MANNER',
       'WEATHERIMPROPER BACKING',
       'DISREGARDING STOP SIGNDISREGARDING STOP SIGN',
       'IMPROPER BACKINGUNDER THE INFLUENCE OF ALCOHOL/DRUGS (USE WHEN ARREST IS EFFECTED)',
       'OPERATING VEHICLE IN ERRATIC, RECKLESS, CARELESS, 

In [88]:
reordered_columns = [
    'CRASH_DATE',
    'DATE_POLICE_NOTIFIED',

    'ON_SCENE',
    'HIT_AND_RUN',

    'POSTED_SPEED_LIMIT',

    'TRAFFIC_CONTROL_DEVICE',
    'DEVICE_FUNCTIONING',  # Todo: Change to TRAFFIC_DEVICE_FUNCTIONING

    'WEATHER_CONDITION',
    'LIGHTING_CONDITION',
    'ROADWAY_SURFACE_COND',
    'ROAD_DEFECT',

    'TRAFFICWAY_TYPE',
    'ALIGNMENT',  # Todo: Change to ROAD_LEVEL

    'DAMAGE',  # Todo: Change to DAMAGE_AMT

    'CRASH_TYPE',

    'PRIM_CONTRIBUTORY_CAUSE',
    'SEC_CONTRIBUTORY_CAUSE',

    'REPORTABLE',
    'INJURIES_TOTAL',
    'INJURIES_FATAL',
    'INJURIES_INCAPACITATING',
    'INJURIES_NON_INCAPACITATING',
    'INJURIES_REPORTED_NOT_EVIDENT',
    'INJURIES_NO_INDICATION',
    'INJURIES_UNKNOWN',

    'LATITUDE',
    'LONGITUDE',
]
df_clean_data = df_cp4[reordered_columns]

In [89]:
df_clean_data

,CRASH_DATE,DATE_POLICE_NOTIFIED,ON_SCENE,HIT_AND_RUN,POSTED_SPEED_LIMIT,TRAFFIC_CONTROL_DEVICE,DEVICE_FUNCTIONING,WEATHER_CONDITION,LIGHTING_CONDITION,ROADWAY_SURFACE_COND,...,REPORTABLE,INJURIES_TOTAL,INJURIES_FATAL,INJURIES_INCAPACITATING,INJURIES_NON_INCAPACITATING,INJURIES_REPORTED_NOT_EVIDENT,INJURIES_NO_INDICATION,INJURIES_UNKNOWN,LATITUDE,LONGITUDE
0,2025-01-14 12:25:00,2025-01-14 12:38:00,1,1,30,TRAFFIC SIGNAL,True,SNOW,DAYLIGHT,SNOW OR SLUSH,...,False,0,0,0,0,0,2,0.0,41.997808,-87.655770
1,2025-05-23 09:30:00,2025-05-27 10:40:00,0,1,30,STOP SIGN/FLASHER,False,UNKNOWN,DAYLIGHT,UNKNOWN,...,False,0,0,0,0,0,2,0.0,41.946529,-87.688106
2,2025-04-05 20:00:00,2025-04-05 21:23:00,1,1,30,NO CONTROLS,False,CLEAR,UNKNOWN,DRY,...,False,0,0,0,0,0,1,0.0,41.899325,-87.715074
3,2025-05-23 09:15:00,2025-05-23 09:16:00,1,-1,30,NO CONTROLS,False,CLEAR,DAYLIGHT,DRY,...,True,1,0,0,1,0,2,0.0,41.902793,-87.699412
4,2025-01-14 08:00:00,2025-01-14 08:45:00,0,-1,30,TRAFFIC SIGNAL,True,SNOW,DAYLIGHT,WET,...,False,0,0,0,0,0,2,0.0,41.691207,-87.720555
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
984007,2025-08-07 18:40:00,2025-08-08 15:30:00,0,-1,30,TRAFFIC SIGNAL,True,CLEAR,DAYLIGHT,DRY,...,False,0,0,0,0,0,2,0.0,42.012120,-87.690100
984008,2025-08-16 23:50:00,2025-08-16 23:55:00,1,1,30,NO CONTROLS,False,UNKNOWN,"DARKNESS, LIGHTED ROAD",UNKNOWN,...,True,0,0,0,0,0,2,0.0,41.778225,-87.747206
984009,2025-08-17 14:00:00,2025-08-18 15:30:00,0,-1,30,STOP SIGN/FLASHER,False,CLEAR,DAYLIGHT,DRY,...,False,0,0,0,0,0,2,0.0,41.838002,-87.641182
984010,2025-08-18 00:08:00,2025-08-18 00:09:00,-1,-1,30,NO CONTROLS,False,CLEAR,"DARKNESS, LIGHTED ROAD",DRY,...,True,0,0,0,0,0,1,0.0,41.904205,-87.678698


In [90]:
export_path = library.path.get_cleaned_path(path)
df_clean_data.to_csv(export_path)